In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
# To use the web search tool a schema needs to be provided to Claude
# The max_uses field limits how many searches Claude can perform - number of research times
# Access to the web can be restricted by using the "allowed_domains" property

web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search", 
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [9]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(messages, tools=[web_search_schema])
response

Message(id='msg_015wfgTmHqszqQx5zHjBt8cG', container=None, content=[TextBlock(citations=None, text='The best exercises for gaining leg muscle are compound movements that work multiple muscle groups simultaneously. Here are the top choices:\n\n**1. Barbell Back Squat**\nThis is often considered the king of leg exercises. It targets the quadriceps, hamstrings, glutes, and calves while also engaging your core. Squats allow you to lift heavy weights progressively, which is key for muscle growth.\n\n**2. Romanian Deadlifts (RDLs)**\nExcellent for building the posterior chain, particularly the hamstrings and glutes. This exercise complements squats by targeting muscles that squats work less directly.\n\n**3. Bulgarian Split Squats**\nA unilateral (single-leg) exercise that\'s highly effective for building muscle while also addressing strength imbalances between legs. It heavily targets quads and glutes.\n\n**4. Leg Press**\nAllows you to move heavy weight safely while targeting quads, hamstr